# Claude 3.7 Sonnet의 병렬 도구 호출

Claude 3.7 Sonnet은 `disable_parallel_tool_use`를 설정하지 않았더라도 응답에서 도구를 병렬로 호출할 가능성이 낮을 수 있습니다. 이를 우회하려면 다른 도구들의 호출을 한꺼번에 감싸는 메타 도구 역할의 "배치 도구(batch tool)"를 도입하는 것을 권장합니다. 이 도구가 있으면 모델이 이를 사용해 여러 도구를 동시에 병렬로 호출한다는 것을 확인했습니다.

문제 상황을 살펴보고 이 우회 방법을 자세히 들여다보겠습니다.

In [ ]:
from anthropic import Anthropic

client = Anthropic()
MODEL_NAME = "claude-sonnet-4-6"

## 여러 도구 호출이 필요한 질의 수행하기

기본 동작에서는 Claude에 병렬 도구 호출이 허용된다는 점을 떠올려 보세요. 기본값인 `tool_choice`의 `auto`와 결합하면, Claude는 지정된 도구 중 무엇이든 호출할 수 있고 한 번의 어시스턴트 턴에서 여러 개를 호출할 수도 있습니다.

Claude에 `get_weather`와 `get_time` 도구를 설정해 보겠습니다.

In [3]:
def get_weather(location):
    # Pretend to get the weather, and just return a fixed value.
    return f"The weather in {location} is 72 degrees and sunny."


def get_time(location):
    # Pretend to get the time, and just return a fixed value.
    return f"The time in {location} is 12:32 PM."


weather_tool = {
    "name": "get_weather",
    "description": "Gets the weather for in a given location",
    "input_schema": {
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "The city and state, e.g. San Francisco, CA",
            },
        },
        "required": ["location"],
    },
}

time_tool = {
    "name": "get_time",
    "description": "Gets the time in a given location",
    "input_schema": {
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "The city and state, e.g. San Francisco, CA",
            },
        },
        "required": ["location"],
    },
}


def process_tool_call(tool_name, tool_input):
    if tool_name == "get_weather":
        return get_weather(tool_input["location"])
    elif tool_name == "get_time":
        return get_time(tool_input["location"])
    else:
        raise ValueError(f"Unexpected tool name: {tool_name}")

다음으로 이 도구들을 Claude에 제공하고 질의를 수행해 보겠습니다.

In [4]:
def make_query_and_print_result(messages, tools=None):
    response = client.messages.create(
        model=MODEL_NAME,
        messages=messages,
        max_tokens=1000,
        tool_choice={"type": "auto"},
        tools=tools or [weather_tool, time_tool],
    )

    for block in response.content:
        match block.type:
            case "text":
                print(block.text)
            case "tool_use":
                print(f"Tool: {block.name}({block.input})")
            case _:
                raise ValueError(f"Unexpected block type: {block.type}")

    return response


MESSAGES = [{"role": "user", "content": "What's the weather and time in San Francisco?"}]

response = make_query_and_print_result(MESSAGES)

I'll check the current weather and time in San Francisco for you.
Tool: get_weather({'location': 'San Francisco, CA'})


둘 다 요청했는데도 Claude가 날씨에 대한 도구 호출 하나만 반환한 것이 보이시나요?

날씨 도구를 호출하고 계속 진행하면 어떻게 되는지 살펴보겠습니다.

In [5]:
last_tool_call = response.content[1]

MESSAGES.append({"role": "assistant", "content": response.content})
MESSAGES.append(
    {
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": last_tool_call.id,
                "content": process_tool_call(response.content[1].name, response.content[1].input),
            }
        ],
    }
)

response = make_query_and_print_result(MESSAGES)

Tool: get_time({'location': 'San Francisco, CA'})


이번에는 Claude가 시간을 얻기 위해 두 번째 도구 호출을 한 것을 볼 수 있습니다. 기술적으로는 곧바로 이뤄지긴 했지만, "왕복"이 필요했다는 점에서 낭비일 수 있습니다. 먼저 Claude가 날씨를 요청하면 우리가 처리하고, _그다음_ Claude가 시간을 요청하면 그것을 _또_ 처리해야 하기 때문입니다.

Claude는 결과를 받아 여전히 올바르게 처리하지만, 두 도구를 한 번의 호출에 함께 쓰도록 유도하면 동시에 처리할 수 있어 유리합니다.

## 배치 도구 도입하기

여러 도구 호출을 하나로 묶을 기회를 Claude에 주기 위해 `batch_tool`을 도입해 보겠습니다.

In [6]:
import json

batch_tool = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "types": "object",
                    "properties": {
                        "name": {
                            "types": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "types": "string",
                            "description": "The arguments to the tool",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}


def process_tool_with_maybe_batch(tool_name, tool_input):
    if tool_name == "batch_tool":
        results = []
        for invocation in tool_input["invocations"]:
            results.append(
                process_tool_call(invocation["name"], json.loads(invocation["arguments"]))
            )
        return "\n".join(results)
    else:
        return process_tool_call(tool_name, tool_input)

이제 기존 날씨·시간 도구와 함께 새로 만든 배치 도구를 Claude에 제공하고, 날씨와 시간을 모두 요구하는 질의를 던지면 어떻게 되는지 살펴보겠습니다.

In [7]:
MESSAGES = [{"role": "user", "content": "What's the weather and time in San Francisco?"}]

response = make_query_and_print_result(MESSAGES, tools=[weather_tool, time_tool, batch_tool])

I can help you check both the weather and the time in San Francisco. Let me get that information for you right away.
Tool: batch_tool({'invocations': [{'name': 'get_weather', 'arguments': '{"location": "San Francisco, CA"}'}, {'name': 'get_time', 'arguments': '{"location": "San Francisco, CA"}'}]})


이번에는 Claude가 배치 도구를 사용해 시간과 날씨를 한 번에 질의한 것을 볼 수 있습니다. 덕분에 두 가지를 동시에 처리할 수 있어 최종 결과까지의 전체 지연 시간을 줄일 수 있습니다.

In [8]:
last_tool_call = response.content[1]

MESSAGES.append({"role": "assistant", "content": response.content})
MESSAGES.append(
    {
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": last_tool_call.id,
                "content": process_tool_with_maybe_batch(
                    response.content[1].name, response.content[1].input
                ),
            }
        ],
    }
)

response = make_query_and_print_result(MESSAGES)

Here's the information you requested:

Weather in San Francisco, CA: 72 degrees and sunny
Time in San Francisco, CA: 12:32 PM

Is there anything else you'd like to know about San Francisco?
